In [ ]:
# ============================================================
# HYBRID LSTM-XGBoost STOCK RETURN PREDICTION — UNIVERSAL (v2)
# Retrain: ~100 diverse US tickers | train-only GLOBAL target norm
# Artifacts are drop-in compatible with the FastAPI service (main.py)
# ============================================================

!pip install -q xgboost yfinance scikit-learn==1.5.2

import os, math, time, json, pickle
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from warnings import simplefilter
simplefilter(action='ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Setup complete. Using device: {device}')

In [ ]:
# ============================================================
# UNIVERSAL CONFIG & TICKER UNIVERSE (~100 diverse US large-caps)
# Deliberately includes decliners (INTC, BA, PFE, PYPL, PARA, T ...)
# so the model sees downside, not just bull-market survivors.
# ============================================================

LOOK_BACK  = 60
HORIZON    = 30                       # deployed prediction horizon (trading days)
START, END = '2015-01-01', '2026-01-01'

LSTM_INPUT_COLS = ['Volume_Ratio', 'Return', 'RSI', 'MACD', 'MACD_signal']

TECH_COLS = [
    'SMA_5_Ratio', 'SMA_10_Ratio', 'SMA_15_Ratio', 'SMA_30_Ratio',
    'EMA_9_Ratio', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
    'Volatility_20', 'Momentum_10', 'Momentum_21',
    'Volume_Change', 'Volume_Ratio'
]

TICKERS = [
    # Tech / Semis / Communications
    'AAPL','MSFT','NVDA','AMZN','GOOGL','META','TSLA','AVGO','ORCL','AMD',
    'INTC','QCOM','TXN','MU','AMAT','ADBE','CRM','NOW','INTU','CSCO',
    'IBM','PYPL','UBER','NFLX','DIS','CMCSA','T','VZ','TMUS','PARA','WBD',
    # Financials
    'JPM','BAC','WFC','GS','MS','C','AXP','BLK','SCHW','V','MA',
    # Healthcare
    'UNH','JNJ','LLY','ABBV','MRK','PFE','TMO','ABT','DHR','BMY','AMGN','GILD','CVS','MDT',
    # Energy
    'XOM','CVX','COP','SLB','EOG','MPC','PSX','VLO','OXY',
    # Consumer
    'WMT','COST','PG','KO','PEP','MCD','NKE','SBUX','TGT','HD','LOW','BKNG','MDLZ','CL',
    # Industrials
    'CAT','DE','HON','UPS','RTX','LMT','GE','MMM','BA','FDX','EMR','ETN',
    # Materials
    'LIN','SHW','FCX','NEM',
    # Real estate / Utilities
    'AMT','PLD','SPG','O','WELL','SO','DUK','NEE',
]
TICKERS = sorted(set(TICKERS))
print(f'{len(TICKERS)} tickers in universe.')

In [ ]:
# ============================================================
# DOWNLOAD OHLCV — ALL TICKERS (skip any that fail / are too short)
# ============================================================

stock_dfs = {}
failed    = []

for i, ticker in enumerate(TICKERS, 1):
    try:
        df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
        if df is None or len(df) < 400:
            failed.append(ticker); continue
        df = df.reset_index()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.columns = [str(c).capitalize() for c in df.columns]
        df.rename(columns={'Adj close': 'Close'}, inplace=True)
        df = df[['Date','Open','High','Low','Close','Volume']].copy()
        df = df.sort_values('Date').reset_index(drop=True).ffill().bfill()
        stock_dfs[ticker] = df
    except Exception:
        failed.append(ticker)
    if i % 20 == 0:
        print(f'  ...{i}/{len(TICKERS)} processed')

print(f'\nDownloaded {len(stock_dfs)} tickers | failed/skipped: {failed}')

In [ ]:
# ============================================================
# FEATURE ENGINEERING  (identical to main.py compute_features)
#   - MACD normalized by Close
#   - SMA/EMA as shifted price ratios (no look-ahead)
# Plus the RAW 30-day forward target (normalized later, train-only).
# ============================================================

def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.where(delta > 0, 0.0)
    loss  = (-delta).where(delta < 0, 0.0)
    rs    = gain.rolling(period).mean() / loss.rolling(period).mean()
    return 100.0 - (100.0 / (1.0 + rs))

def compute_features(df):
    df = df.copy()
    close, volume = df['Close'], df['Volume']

    # --- LSTM input features (5) ---
    df['Return']       = close.pct_change()
    df['Volume_Ratio'] = volume / volume.rolling(20).mean()
    df['RSI']          = compute_rsi(close, 14)
    ema_12 = close.ewm(span=12, min_periods=12).mean()
    ema_26 = close.ewm(span=26, min_periods=26).mean()
    df['MACD']        = (ema_12 - ema_26) / close
    df['MACD_signal'] = df['MACD'].ewm(span=9, min_periods=9).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

    # --- Tech indicators (14), price-ratio based ---
    for w in [5, 10, 15, 30]:
        df[f'SMA_{w}_Ratio'] = (close.rolling(w).mean().shift() / close).fillna(1.0)
    df['EMA_9_Ratio']   = (close.ewm(span=9).mean().shift() / close).fillna(1.0)
    df['Volatility_20'] = close.pct_change().rolling(20).std()
    df['Momentum_10']   = close.pct_change(periods=10)
    df['Momentum_21']   = close.pct_change(periods=21)
    df['Volume_Change'] = volume.pct_change()
    df['Volume_Ratio']  = df['Volume_Ratio'].replace([np.inf, -np.inf], np.nan)

    # --- RAW target: forward 30-day cumulative return ---
    df['Target_30d'] = df['Return'].rolling(HORIZON).sum().shift(-HORIZON)
    return df

feat_dfs = {}
need = list(set(LSTM_INPUT_COLS + TECH_COLS + ['Close', 'Target_30d']))
for t, df in stock_dfs.items():
    d = compute_features(df)
    d[TECH_COLS] = d[TECH_COLS].replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=need).reset_index(drop=True)   # only require 30d target -> keeps ~330 more rows/ticker
    if len(d) > LOOK_BACK + 50:
        feat_dfs[t] = d

total_rows = sum(len(d) for d in feat_dfs.values())
print(f'{len(feat_dfs)} tickers usable | {total_rows:,} clean rows total')

In [ ]:
# ============================================================
# PER-STOCK CHRONOLOGICAL SPLIT (70 / 15 / 15) — time-based
# ============================================================
TRAIN_RATIO, VALID_RATIO = 0.70, 0.15
splits = {}
for t, d in feat_dfs.items():
    n = len(d)
    splits[t] = (int(n * TRAIN_RATIO), int(n * (TRAIN_RATIO + VALID_RATIO)))
print('Split indices computed for', len(splits), 'tickers.')

In [ ]:
# ============================================================
# GLOBAL SCALING (universal across all tickers) — FIT ON TRAIN ONLY
#   + GLOBAL train-only target normalization (mean/std of raw Target_30d)
#
# KEY FIX: the forward normalization here uses the SAME global mean/std
# that we store in target_stats.json, so main.py's denormalization
#       raw_pred = z * std + mean
# is now an EXACT inverse (no per-ticker / look-ahead mismatch).
# ============================================================

global_feature_scaler = MinMaxScaler(feature_range=(-1, 1))
global_tech_scaler    = MinMaxScaler(feature_range=(-1, 1))

ftr, ttr, ytr_raw = [], [], []
for t, d in feat_dfs.items():
    te = splits[t][0]
    ftr.append(d[LSTM_INPUT_COLS].iloc[:te])
    ttr.append(d[TECH_COLS].iloc[:te])
    ytr_raw.append(d['Target_30d'].iloc[:te].values)

global_feature_scaler.fit(pd.concat(ftr))
global_tech_scaler.fit(pd.concat(ttr))

ytr_raw     = np.concatenate(ytr_raw)
TARGET_MEAN = float(ytr_raw.mean())
TARGET_STD  = float(ytr_raw.std())
print(f'Global TRAIN target: mean={TARGET_MEAN:.6f}  std={TARGET_STD:.6f}')
print('  (this is the raw 30d-return distribution the model is anchored to)')

In [ ]:
# ============================================================
# BUILD LSTM WINDOWS + ALIGNED TECH / TARGET ARRAYS PER SPLIT
# Target is globally z-normalized with TRAIN-only mean/std.
# ============================================================

def build_arrays(which):
    seqs, techs, ys, labs, closes = [], [], [], [], []
    for t, d in feat_dfs.items():
        te, ve = splits[t]
        fs = global_feature_scaler.transform(d[LSTM_INPUT_COLS].values).astype('float32')
        ts = global_tech_scaler.transform(d[TECH_COLS].values).astype('float32')
        yv = ((d['Target_30d'].values - TARGET_MEAN) / TARGET_STD).astype('float32')
        cl = d['Close'].values
        idx = np.arange(LOOK_BACK - 1, len(d))
        if   which == 'train': idx = idx[idx < te]
        elif which == 'valid': idx = idx[(idx >= te) & (idx < ve)]
        else:                  idx = idx[idx >= ve]
        if len(idx) == 0:
            continue
        seqs.append(np.stack([fs[i - LOOK_BACK + 1:i + 1] for i in idx]))
        techs.append(ts[idx]); ys.append(yv[idx])
        labs.append(np.array([t] * len(idx))); closes.append(cl[idx])
    return (np.concatenate(seqs), np.concatenate(techs),
            np.concatenate(ys), np.concatenate(labs), np.concatenate(closes))

x_train, tech_train, y_train, lab_train, close_train = build_arrays('train')
x_valid, tech_valid, y_valid, lab_valid, close_valid = build_arrays('valid')
x_test,  tech_test,  y_test,  lab_test,  close_test  = build_arrays('test')

print('LSTM input :', x_train.shape, x_valid.shape, x_test.shape)
print('Tech       :', tech_train.shape)
print('Target y   : %s | mean=%.3f std=%.3f' % (y_train.shape, y_train.mean(), y_train.std()))

In [ ]:
# ============================================================
# LSTM FEATURE EXTRACTOR  (matches main.py LSTMBackbone exactly)
#   features = out[:, -1, :]   (last time-step output, NOT h_n[-1])
#   explicit zero h0 / c0
# ============================================================

class LSTMFeatureExtractor(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout=0.5):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        features   = out[:, -1, :]
        prediction = self.fc(features)
        return prediction, features

INPUT_DIM, HIDDEN_DIM, NUM_LAYERS = 5, 64, 2
model = LSTMFeatureExtractor(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, dropout=0.5).to(device)
print(model)
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ============================================================
# TRAIN LSTM  (target = globally z-normalized 30d return)
#   No second MinMax on the target -> single, consistent scaling.
# ============================================================

NUM_EPOCHS, LR, PATIENCE, BATCH_SIZE = 300, 5e-4, 40, 128

xtr = torch.tensor(x_train, device=device)
ytr = torch.tensor(y_train, device=device).unsqueeze(1)
xva = torch.tensor(x_valid, device=device)
yva = torch.tensor(y_valid, device=device).unsqueeze(1)

loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=8)
loader    = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(xtr, ytr), batch_size=BATCH_SIZE, shuffle=True)

history = {'train': [], 'valid': []}
best, best_state, no_improve, best_epoch = float('inf'), None, 0, 0
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    tl = []
    for bx, by in loader:
        optimizer.zero_grad()
        pred, _ = model(bx)
        loss = loss_fn(pred, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl.append(loss.item())
    model.eval()
    with torch.no_grad():
        vp, _ = model(xva)
        vl = loss_fn(vp, yva).item()
    scheduler.step(vl)
    history['train'].append(float(np.mean(tl))); history['valid'].append(vl)
    if vl < best:
        best, best_epoch = vl, epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0; mark = '  <- best'
    else:
        no_improve += 1; mark = ''
    if epoch % 10 == 0 or mark:
        print(f'  epoch {epoch:>3d} | train {np.mean(tl):.5f} | valid {vl:.5f}{mark}')
    if no_improve >= PATIENCE:
        print(f'  early stop @ epoch {epoch}'); break

model.load_state_dict(best_state); model.eval()
print(f'\nDone in {time.time() - t0:.0f}s | best epoch {best_epoch} | best valid {best:.5f}')

plt.figure(figsize=(9, 4))
plt.plot(history['train'], label='train'); plt.plot(history['valid'], label='valid')
plt.axvline(best_epoch, color='k', ls='--', alpha=.5)
plt.legend(); plt.title('LSTM training loss'); plt.xlabel('epoch'); plt.show()

In [ ]:
# ============================================================
# EXTRACT 64-DIM LSTM EMBEDDINGS -> HYBRID MATRIX
#   X_hybrid = [64 LSTM features | 14 tech indicators] = 78 features
# ============================================================

def extract(x):
    out = []
    with torch.no_grad():
        for i in range(0, len(x), 4096):
            _, f = model(torch.tensor(x[i:i + 4096], device=device))
            out.append(f.cpu().numpy())
    return np.concatenate(out)

F_train, F_valid, F_test = extract(x_train), extract(x_valid), extract(x_test)
X_train = np.concatenate([F_train, tech_train], axis=1)
X_valid = np.concatenate([F_valid, tech_valid], axis=1)
X_test  = np.concatenate([F_test,  tech_test],  axis=1)
print('Hybrid feature matrix:', X_train.shape, '(expect 78 columns)')

In [ ]:
# ============================================================
# HYBRID XGBOOST — narrowed grid around the known-good params
# (the data + normalization were the problem, not the hyperparameters)
# ============================================================

param_grid = {
    'n_estimators':     [200, 300],
    'learning_rate':    [0.005, 0.01],
    'max_depth':        [4, 5],
    'min_child_weight': [10, 20],
    'subsample':        [0.6],
    'colsample_bytree': [0.6],
    'gamma':            [0.1],
    'random_state':     [42],
}

print('GridSearchCV (16 combos x 3-fold = 48 fits)...')
t0 = time.time()
search = GridSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', tree_method='hist'),
    param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
print(f'\nDone in {time.time() - t0:.0f}s')
print('Best params:', search.best_params_)
print('CV RMSE    :', math.sqrt(-search.best_score_))

xgb_best = xgb.XGBRegressor(**search.best_params_, objective='reg:squarederror')
xgb_best.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

In [ ]:
# ============================================================
# EVALUATION — normalized metrics + DENORMALIZED return distribution
# This is where we confirm the 'all positive / huge / conf=1' symptom
# is actually fixed.
# ============================================================

def evaluate(name, yt, yp):
    rmse    = math.sqrt(mean_squared_error(yt, yp))
    r2      = r2_score(yt, yp)
    dir_acc = np.mean(np.sign(yt) == np.sign(yp)) * 100
    print(f'  {name:<7s} RMSE {rmse:.4f} | R2 {r2:+.4f} | DirAcc {dir_acc:.1f}%')

p_train = xgb_best.predict(X_train)
p_valid = xgb_best.predict(X_valid)
p_test  = xgb_best.predict(X_test)

print('NORMALIZED (z-score) hybrid metrics:')
evaluate('train', y_train, p_train)
evaluate('valid', y_valid, p_valid)
evaluate('test',  y_test,  p_test)

# --- denormalize back to real 30d returns (SAME formula as main.py) ---
real_test = p_test * TARGET_STD + TARGET_MEAN
print('\nDENORMALIZED test predictions (real 30d return %):')
print(f'  min {real_test.min()*100:+.2f}%   max {real_test.max()*100:+.2f}%   '
      f'mean {real_test.mean()*100:+.2f}%   std {real_test.std()*100:.2f}%')
print(f'  predicted UP: {(real_test > 0).mean()*100:.1f}%    '
      f'DOWN: {(real_test <= 0).mean()*100:.1f}%')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(real_test * 100, bins=60, color='steelblue', edgecolor='k', alpha=.7)
ax[0].axvline(0, color='r', ls='--'); ax[0].set_title('Denormalized test predictions (30d %)')
ax[0].set_xlabel('predicted return %')
ax[1].scatter(y_test, p_test, s=6, alpha=.3)
lims = [min(y_test.min(), p_test.min()), max(y_test.max(), p_test.max())]
ax[1].plot(lims, lims, 'k--', alpha=.5)
ax[1].set_xlabel('actual (z)'); ax[1].set_ylabel('pred (z)'); ax[1].set_title('Pred vs Actual')
plt.tight_layout(); plt.show()

## Model Comparison — Hybrid vs Standalone LSTM vs Standalone XGBoost

Reproduces the paper's **Table III** methodology on the *upgraded* model: **RMSE, MAE, R², and directional accuracy** for all three models on the held-out test set.

- **LSTM-Only** — the LSTM's own linear head prediction.
- **XGBoost-Only** — XGBoost on the 14 technical indicators alone (no LSTM embeddings), independently grid-searched.
- **Hybrid** — XGBoost on the 78-dim `[LSTM embedding | tech]` matrix.

Metrics are computed on **denormalized 30-day returns** (real units, same scale as the paper), and directional accuracy is measured relative to 0 (UP vs DOWN), exactly as `main.py` decides direction. Run the cells from the top once, then these three cells.

In [ ]:
# ============================================================
# BASELINES FOR COMPARISON — Standalone LSTM & Standalone XGBoost
#   Standalone LSTM : the LSTM's own linear head (fc) output, learned
#                     jointly during LSTM training (z-normalized target).
#   Standalone XGB  : XGBoost on the 14 technical indicators ONLY
#                     (no LSTM embeddings), independently grid-searched.
#   Hybrid          : xgb_best on the 78-dim [LSTM | tech] matrix.
#   All three share the same split + target normalization, so the
#   metrics in the next cell compare 1:1 (paper Table III methodology).
# ============================================================

# --- Standalone LSTM predictions (z-score) ---
def lstm_predict(x):
    out = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(x), 4096):
            pred, _ = model(torch.tensor(x[i:i + 4096], device=device))
            out.append(pred.cpu().numpy().ravel())
    return np.concatenate(out)

lstm_pred_test = lstm_predict(x_test)

# --- Standalone XGBoost on technical indicators only (independent grid search) ---
print('GridSearchCV for standalone XGBoost (14 tech indicators only)...')
t0 = time.time()
search_tech = GridSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', tree_method='hist'),
    param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=0)
search_tech.fit(tech_train, y_train)
print(f'  done in {time.time() - t0:.0f}s | best params: {search_tech.best_params_}')

xgb_tech = xgb.XGBRegressor(**search_tech.best_params_, objective='reg:squarederror')
xgb_tech.fit(tech_train, y_train, eval_set=[(tech_valid, y_valid)], verbose=False)
xgb_tech_pred_test = xgb_tech.predict(tech_test)

# --- Hybrid predictions (z-score) ---
hybrid_pred_test = xgb_best.predict(X_test)

print('Baseline predictions ready: LSTM-Only, XGBoost-Only, Hybrid.')

In [ ]:
# ============================================================
# METRICS — RMSE, MAE, R2, Directional Accuracy
#   Reported on DENORMALIZED 30-day returns (real units, same scale
#   as the paper). Directional accuracy is measured relative to 0
#   (UP vs DOWN) — exactly how main.py decides direction — NOT relative
#   to the z-score mean, so it reflects real deployed behaviour.
# ============================================================

def denorm(z):
    return z * TARGET_STD + TARGET_MEAN

y_test_real = denorm(y_test)

def compute_metrics(name, yp_z):
    yp_real = denorm(yp_z)
    return {
        'Model':         name,
        'RMSE':          math.sqrt(mean_squared_error(y_test_real, yp_real)),
        'MAE':           mean_absolute_error(y_test_real, yp_real),
        'R2':            r2_score(y_test_real, yp_real),
        'Dir. Acc. (%)': np.mean(np.sign(y_test_real) == np.sign(yp_real)) * 100,
    }

results = pd.DataFrame([
    compute_metrics('LSTM-Only',    lstm_pred_test),
    compute_metrics('XGBoost-Only', xgb_tech_pred_test),
    compute_metrics('Hybrid',       hybrid_pred_test),
])

print('\n30-DAY HORIZON — GLOBAL TEST SET (denormalized real returns)\n')
print(results.to_string(index=False, formatters={
    'RMSE':          '{:.4f}'.format,
    'MAE':           '{:.4f}'.format,
    'R2':            '{:+.4f}'.format,
    'Dir. Acc. (%)': '{:.1f}'.format,
}))

results.to_csv('model_comparison_metrics.csv', index=False)
print('\nSaved -> model_comparison_metrics.csv')

In [ ]:
# ============================================================
# COMPARISON CHART (for the thesis presentation)
#   3 panels: RMSE, MAE, Directional Accuracy across the 3 models.
#   Saves a PNG you can drop straight into the slides.
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
mdls   = results['Model'].tolist()
colors = ['#8A9099', '#B57A00', '#FFB000']   # Quant Terminal palette

panels = [
    ('RMSE',          'RMSE  (lower = better)',                    '{:.4f}'),
    ('MAE',           'MAE  (lower = better)',                     '{:.4f}'),
    ('Dir. Acc. (%)', 'Directional Accuracy %  (higher = better)', '{:.1f}'),
]
for axi, (col, title, fmt) in zip(ax, panels):
    bars = axi.bar(mdls, results[col], color=colors, edgecolor='k')
    axi.set_title(title)
    axi.bar_label(bars, labels=[fmt.format(v) for v in results[col]], padding=3)
    axi.grid(axis='y', alpha=.3)
    axi.margins(y=0.15)

plt.suptitle('Hybrid vs Standalone Models — 30-day horizon (test set)',
             y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> model_comparison.png')

In [ ]:
# ============================================================
# SAVE ARTIFACTS — drop-in replacements for  main.py /models
# Same 6 files, same interface. Replace and restart the service.
# ============================================================

OUT = 'universal_hybrid_model'
os.makedirs(OUT, exist_ok=True)

torch.save(model.state_dict(), f'{OUT}/lstm_backbone.pth')
xgb_best.save_model(f'{OUT}/xgb_head.json')
with open(f'{OUT}/global_feature_scaler.pkl', 'wb') as f: pickle.dump(global_feature_scaler, f)
with open(f'{OUT}/global_tech_scaler.pkl',    'wb') as f: pickle.dump(global_tech_scaler, f)
with open(f'{OUT}/target_stats.json', 'w') as f:
    json.dump({'mean': TARGET_MEAN, 'std': TARGET_STD}, f, indent=2)
with open(f'{OUT}/universal_config.json', 'w') as f:
    json.dump({
        'look_back':   LOOK_BACK,
        'lstm_params': {'input_dim': INPUT_DIM, 'hidden_dim': HIDDEN_DIM, 'layers': NUM_LAYERS},
        'feature_cols': LSTM_INPUT_COLS,
        'tech_cols':    TECH_COLS,
    }, f)

import shutil
shutil.make_archive('universal_hybrid_model', 'zip', OUT)
print('Saved + zipped -> universal_hybrid_model.zip')
print('Replace the 6 files in  Market Prediction/models/  with these, then restart uvicorn.')
try:
    from google.colab import files
    files.download('universal_hybrid_model.zip')
except Exception:
    pass